In [1]:
import os
import sys
import time
experiments_dir = os.path.abspath('..')
if experiments_dir not in sys.path:
    sys.path.append(experiments_dir)
    
from ingestion.ingestion_pipeline_2 import IngPipeline
from rag.rag_pipeline_2 import RagPipeline

h:\projects\ai_based\Agent-Factory\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
h:\projects\ai_based\Agent-Factory\.venv\Lib\site-packages\transformers\utils\hub.py:106: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(


http://192.168.1.69:8000/generate


In [2]:
ing_configs = {
    'chunk_size': 200,
    'embed_model': "intfloat/e5-large-v2",
    'chunking_approach': "recursive",
    'kbs_path': "../test_data",
    "ingest_pip_version": "2.0",
    "embed_table": "embeddings_table_v2_0"
}

rag_configs = {
    'system_prompt_rag': "You are a helpful assistant that provides accurate and concise answers based on the provided context.",
    'embeddings_model_id': "intfloat/e5-large-v2",
    'cross_encoder_id': "cross-encoder/ms-marco-MiniLM-L-6-v2",
    'top_k': 5,
    'temperature': 0,
    'ce_threshold': 0,
    'search_type': "cosine",
    'src': "test_source",
    'embed_table': "embeddings_table_v2_0", # ingest synch param
    "ingest_pip_version": "2.0",            # ingest synch param
    'rag_pip_version': "2.0",               # NOT YET SUPPORTED
    "embed_table": "embeddings_table_v2_0", # ingest synch param
    'chunk_size': 200,                      # ingest synch param
}

global_configs = {
    'chunking_approach': "recursive",
    'system_prompt_rag': "You are a helpful assistant that provides accurate and concise answers based on the provided context.",
    'embeddings_model_id': "intfloat/e5-large-v2",
    'cross_encoder_id': "cross-encoder/ms-marco-MiniLM-L-6-v2",
    'top_k': 5,
    'temperature': 0,
    'ce_threshold': 0,
    'search_type': "cosine",
    'src': "test_source",
    'embed_table': "embeddings_table_v2_0", # ingest synch param
    "ingest_pip_version": "2.0",            # ingest synch param
    'rag_pip_version': "2.0",               # NOT YET SUPPORTED
    "embed_table": "embeddings_table_v2_0", # ingest synch param
    'chunk_size': 200,                      # ingest synch param
}

db_database = os.getenv("DB_DATABASE")
db_user = os.getenv("DB_USER")
db_password = os.getenv("DB_PASSWORD")
db_host = os.getenv("DB_HOST")

DB_CONFIG = {
    "dbname": db_database,
    "user": db_user,
    "password": db_password,
    "host": db_host
}

In [3]:
ing_pipeline = IngPipeline(DB_CONFIG, ing_configs)
rag_pipeline = RagPipeline(DB_CONFIG, rag_configs)

DEBUG: Initialized IngPipeline with kbs_path='../test_data', embed_model_id='intfloat/e5-large-v2', chunk_size=200, chunking_approach='recursive'


In [4]:
ing_pipeline.ingest_pipeline()

DEBUG: Executed query to fetch unique documents
DEBUG: Found 2 unique documents
[('Mykola_Shumskiy_Employment_Agreement-1.pdf', 'intfloat/e5-large-v2', 400, 400, 'recursive', '2.0'), ('Mykola_Shumskiy_Employment_Agreement-1.pdf', 'intfloat/e5-large-v2', 200, 200, 'recursive', '2.0')]
DEBUG: Knowledge base path exists: ../test_data
Starting ingestion pipeline...
DEBUG: Processing knowledge base: test_kb at path ../test_data\test_kb

Processing knowledge base: test_kb
Checking document: Mykola_Shumskiy_Employment_Agreement-1.pdf
  -> Mykola_Shumskiy_Employment_Agreement-1.pdf already ingested with the same configuration.


In [5]:
def run_rag(user_prompt,llm):
    t0 = time.time()
    rag_output = rag_pipeline.generate_rag(
        user_prompt = user_prompt,
        selected_kb = 'test_kb',
        llm = "gemma3n:e2b")
    t1 = time.time()
    t_rag = t1 - t0
    return rag_output,t_rag,llm

def process_rag_output(rag_output,user_prompt,llm,t_rag):

    chunks = []
    for i, chunk in enumerate(rag_output['selected_chunks']):
        chunk = {
            "document": chunk['document'],
            "pages": chunk['pages'],
            "text": chunk['text'],
            "document": chunk['document'],
            "similarity_score": chunk['similarity'],
            "ce_score": chunk['ce_score'],
        }
        chunks.append(chunk)

    output = {
        "user_prompt": user_prompt,
        "response_text": rag_output['response']['message']['content'],
        "document_&_pages":rag_output['document_pages'],
        "chunks": chunks,
        "t_semantic_search": rag_output['t_semantic_search'],
        "t_process_context": rag_output['t_process_context'],
        "rag_time": t_rag,
        "llm": llm,
        }
    return output

def run_experiment(user_prompt, llm):
    rag_output, t_rag, llm = run_rag(user_prompt, llm)
    output = process_rag_output(rag_output, user_prompt, llm,t_rag)
    return output

In [6]:
user_prompt = "What the compensation in the contract?"
llm = "gemma3n:e2b"

output = run_experiment(user_prompt, llm)
output

intfloat/e5-large-v2
Retrieving embeddings...
Performing semantic search...
Unloading embeddings model...
Processing context...
Processing references...
Calling LLMP...
http://192.168.1.69:8000/generate


{'user_prompt': 'What the compensation in the contract?',
 'response_text': 'The gross monthly salary is € 3,571.42 (three thousand five hundred and seventy -one euros and forty -two cents). This is subject to legal deductions.\n\nAdditionally, the remuneration includes:\n\n*   Vacation and Christmas subsidy\n*   Daily food allowance of €7.63\n*   Other prizes or bonuses (subject to legal deductions)\n\nThe contract also states that the remuneration includes a contribution for the costs of consumption and use of work tools.',
 'document_&_pages': {'Mykola_Shumskiy_Employment_Agreement-1.pdf': [1,
   2,
   3,
   5,
   6,
   9]},
 'chunks': [{'document': 'Mykola_Shumskiy_Employment_Agreement-1.pdf',
   'pages': [9],
   'text': 'Regulation EU 2016/679, of the European Parliament and of the Council, of April 27, under the terms of this clause, which is supplemented by the Information Statement signed by him/her and attached to this Agreement as ANN EX I. 15ª. - TERMINATION AND DENUNCIATION

In [6]:
print(response[0]['message']['content'])

The gross monthly salary is € 3,571.42 (three thousand five hundred and seventy -one euros and forty -two cents). This is subject to legal deductions.

Additionally, the remuneration includes:

*   Vacation and Christmas subsidy
*   Daily food allowance of €7.63
*   Other prizes or bonuses (subject to legal deductions)

The contract also states that the remuneration includes a contribution for the costs of consumption and use of work tools.


In [ ]:
# Compact display with key information
print("Retrieved Chunks Summary:")
for i, chunk in enumerate(response[2]):
    
    print(f"chunk {i+1}: {chunk['text']}")

Retrieved Chunks Summary:
chunk 1: Regulation EU 2016/679, of the European Parliament and of the Council, of April 27, under the terms of this clause, which is supplemented by the Information Statement signed by him/her and attached to this Agreement as ANN EX I. 15ª. - TERMINATION AND DENUNCIATION 1. The causes of termination of this contract are those foreseen in the Labor Code. 2. If the Employee wish es to terminate the employment contract with the Company¸ must give at least 30 or 60 days notice of his/her intention, depending on whether has up to two years or more than two years of seniority, respectively . 3. If the Employee fails to comply, wholly or partially, with the notice period, he/she shall be obliged to pay the Company compensation equal to the base salary and any per diems corresponding to the missed notice period, without prejudice to civil liability for any damages caused as a result of the failure to comply with the notice period. 4. For the effects of the previous 

In [11]:
# Compact display with key information
print("Retrieved Chunks Summary:")
for i, chunk in enumerate(response[2]):
    print(f"chunk {i+1}: {chunk['text']}")

Retrieved Chunks Summary:
chunk 1: Regulation EU 2016/679, of the European Parliament and of the Council, of April 27, under the terms of this clause, which is supplemented by the Information Statement signed by him/her and attached to this Agreement as ANN EX I. 15ª. - TERMINATION AND DENUNCIATION 1. The causes of termination of this contract are those foreseen in the Labor Code. 2. If the Employee wish es to terminate the employment contract with the Company¸ must give at least 30 or 60 days notice of his/her intention, depending on whether has up to two years or more than two years of seniority, respectively . 3. If the Employee fails to comply, wholly or partially, with the notice period, he/she shall be obliged to pay the Company compensation equal to the base salary and any per diems corresponding to the missed notice period, without prejudice to civil liability for any damages caused as a result of the failure to comply with the notice period. 4. For the effects of the previous 

In [19]:
response[-2][0]

{'id': 30,
 'text': 'Regulation EU 2016/679, of the European Parliament and of the Council, of April 27, under the terms of this clause, which is supplemented by the Information Statement signed by him/her and attached to this Agreement as ANN EX I. 15ª. - TERMINATION AND DENUNCIATION 1. The causes of termination of this contract are those foreseen in the Labor Code. 2. If the Employee wish es to terminate the employment contract with the Company¸ must give at least 30 or 60 days notice of his/her intention, depending on whether has up to two years or more than two years of seniority, respectively . 3. If the Employee fails to comply, wholly or partially, with the notice period, he/she shall be obliged to pay the Company compensation equal to the base salary and any per diems corresponding to the missed notice period, without prejudice to civil liability for any damages caused as a result of the failure to comply with the notice period. 4. For the effects of the previous number, it is 